# 03 · Feature engineering del abandono

**Pregunta:** ¿crear variables nuevas ayuda a anticipar quién se retira? ¿Y de qué variables depende el modelo?

Este notebook sigue el diseño del 02: **foto del asociado** a t0 = corte − 3 meses, con los activos en t0 como
población y `#RETIROS` = 1 si se retira entre t0 y el corte.

**Cómo se mide:** PR-AUC en validación cruzada 5 × 3. Todo lo que aprende de los datos va dentro del pipeline, y
todas las comparaciones usan las mismas particiones.

> **Datos sintéticos.** Las cifras prueban el método, no cómo le irá al modelo con el fondo. El generador decide
> dónde está la señal, así que una variable solo puede ayudar si aprovecha esa regla.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.preprocessing import FunctionTransformer

from proyecto_ml import abandono as a
from proyecto_ml import campeon as c
from proyecto_ml import correlacion as co
from proyecto_ml import tecnicas as t
from proyecto_ml.config import aviso_origen
from proyecto_ml.generador import GeneradorAsociados

SEMILLA, FOLDS, REPETICIONES = 42, 5, 3
COLOR = "#2a78d6"

df, fuente = c.datos_eleccion(origen="sintetico")  # los mismos 5.000 de make campeon
print(aviso_origen("sintetico"))
vista = a.foto(df)
y = vista[a.TARGET]
X = a.preparar(vista)  # las variables de partida, con EDAD y ANTIGUEDAD a t0
CV = RepeatedStratifiedKFold(n_splits=FOLDS, n_repeats=REPETICIONES, random_state=SEMILLA)
BASE_AZAR = y.mean()  # la PR-AUC de un modelo sin información es la tasa de retiro
print(f"{len(X)} asociados en la foto, {y.sum()} se retiran ({BASE_AZAR:.1%}), {X.shape[1]} variables de partida")

## 1. Las variables de partida

Vienen de `abandono.preparar`. Se excluyen las columnas que delatan el resultado (`abandono.FUGA`), porque solo
existen cuando el asociado ya se retiró.

In [ ]:
grupos = {
    "Comportamiento con el fondo (conteos sin fecha)": a.COMPORTAMIENTO,
    "Perfil numérico": ["SALARIO", "ESTRATO", "EDAD", "ANTIGUEDAD"],
    "Perfil categórico": a.CATEGORICAS,
}
pd.DataFrame(
    [{"grupo": g, "columnas": len(cols), "variables": ", ".join(cols)} for g, cols in grupos.items()]
).style.hide(axis="index")

## 2. Las variables que ya creó el proyecto

`abandono.CrearVariables` agrega cuatro. No aprende nada de los datos, así que no hay riesgo de fuga por este paso:

| Variable | Fórmula | Tipo | Idea |
|---|---|---|---|
| `SENALES_SALIDA` | suma de las 5 señales de salida | Agregación | Junta en un número las señales de irse |
| `PRODUCTOS` | ahorros a la vista + contractuales + servicios | Agregación | Cuánto lo ata al fondo |
| `SALIDA_POR_PRODUCTO` | `SENALES_SALIDA / (1 + PRODUCTOS)` | Interacción | Pedir salir teniendo poco que perder |
| `SALARIO_LOG` | `log(1 + SALARIO)` | Transformación | Reduce el sesgo del salario |

In [ ]:
X_nuevas = a.CrearVariables().fit_transform(X)
X_nuevas[a.VARIABLES_NUEVAS].describe().T[["mean", "std", "min", "50%", "max"]].round(2)

### ¿Separan a quienes se retiran?

La tasa de retiro por número de señales de salida muestra si la variable agregada ordena el riesgo. Si la tasa sube
con cada señal adicional, la variable tiene información. Se agrupa desde 5 señales porque hay pocos casos.

In [ ]:
senales = X_nuevas["SENALES_SALIDA"].clip(upper=5).astype(int)
tasa = y.groupby(senales).agg(asociados="size", se_retiran="sum", tasa_retiro="mean").reset_index()
tasa["SENALES_SALIDA"] = tasa["SENALES_SALIDA"].astype(str).replace({"5": "5 o más"})
display(tasa.round(3))
fig = px.bar(
    tasa,
    x="SENALES_SALIDA",
    y="tasa_retiro",
    text=tasa["tasa_retiro"].map("{:.1%}".format),
    hover_data=["asociados", "se_retiran"],
    labels={"SENALES_SALIDA": "Señales de salida", "tasa_retiro": "Tasa de retiro"},
    title="Tasa de retiro según el número de señales de salida (datos sintéticos)",
    color_discrete_sequence=[COLOR],
)
fig.update_xaxes(type="category")
fig.update_yaxes(tickformat=".0%")
fig.update_layout(template="plotly_white")
fig

## 3. Señal y redundancia de cada variable

- **Señal:** relación de cada variable con el retiro (Spearman para numéricas, V de Cramér para categóricas).
- **Redundancia:** dos variables con correlación muy alta dicen casi lo mismo.

Una variable nueva puede tener mucha señal y aun así no aportar, si esa señal ya estaba en las variables de las que
sale. Por eso hay que medirla también dentro del modelo (sección 4).

In [ ]:
senal = co.con_objetivo(X_nuevas, y)
senal["nueva"] = senal["variable"].isin(a.VARIABLES_NUEVAS)
display(senal[["variable", "medida", "valor", "nueva"]].head(12).round(3))
co.pares_redundantes(co.matriz(X_nuevas)).round(3)

## 4. ¿Mejoran al modelo?

Se compara cada opción con su base, partición a partición (`campeon.comparar_particiones`). Una diferencia es
**concluyente** solo si su intervalo del 95 % no toca el 0.

Se prueba con dos modelos porque se comportan distinto: la logística es lineal y solo capta combinaciones que le
demos, mientras que los árboles descubren interacciones por su cuenta.

La base de `tecnicas` usa `class_weight="balanced"`. Por eso sus cifras no coinciden con las del campeón, que no
balancea. Lo que importa aquí es la diferencia dentro de cada fila.

In [ ]:
elegidas = [x for x in t.TECNICAS if x.nombre in ("base", "variables nuevas", "binning")]
tablas = [
    t.comparar(vista.drop(columns=a.TARGET), y, elegidas, clasificador, FOLDS, REPETICIONES, SEMILLA).assign(
        modelo=clasificador
    )
    for clasificador in ("logreg", "hist_gb")
]
comparacion = pd.concat(tablas, ignore_index=True)
comparacion[["modelo", "tecnica", "pr_auc", "dif_pr_auc", "dif_ic_inf", "dif_ic_sup", "concluyente"]].round(3)

### ¿Es real o es ruido?

Con `hist_gb`, las variables nuevas salen concluyentes. `docs/metodologia.md` advierte que un resultado así puede
desaparecer con otros datos, así que se repite con tres datasets nuevos del generador: si solo fuera ruido, el signo
de la diferencia cambiaría de un dataset a otro.

In [ ]:
def con_y_sin(modelo, datos, objetivo):
    """PR-AUC sin y con CrearVariables en las mismas particiones, y la diferencia."""
    sin = a.armar_pipeline(modelo(), random_state=SEMILLA)
    con = a.armar_pipeline(modelo(), crear_variables=True, random_state=SEMILLA)
    r_sin = cross_val_score(sin, datos, objetivo, cv=CV, scoring="average_precision")
    r_con = cross_val_score(con, datos, objetivo, cv=CV, scoring="average_precision")
    d = c.comparar_particiones(r_con, r_sin, FOLDS)
    concluyente = d["ic_inf"] > 0 or d["ic_sup"] < 0
    return {"sin": r_sin.mean(), "con": r_con.mean(), "dif": d["diferencia"], "concluyente": concluyente}


comparados = {
    "hist_gb": lambda: HistGradientBoostingClassifier(class_weight="balanced", random_state=SEMILLA),
    "logística": lambda: LogisticRegression(max_iter=2000),
}
conjuntos = {"desarrollo (semilla 7)": vista}
for semilla in (11, 23, 37):
    conjuntos[f"nuevos (semilla {semilla})"] = a.foto(
        GeneradorAsociados(n=5000, seed=semilla, tasa_retiro=0.1).generar()
    )

filas = [
    {"datos": nombre, "retiros": int(v[a.TARGET].sum()), "modelo": m} | con_y_sin(modelo, a.preparar(v), v[a.TARGET])
    for nombre, v in conjuntos.items()
    for m, modelo in comparados.items()
]
pd.DataFrame(filas).round(3)

### ¿Cuál de las cuatro aporta?

Se agrega una sola variable a la vez al `hist_gb` balanceado, para ver de dónde sale la mejora.

In [ ]:
def con_una(variable):
    """El pipeline de hist_gb con una sola variable de CrearVariables."""
    modelo = HistGradientBoostingClassifier(class_weight="balanced", random_state=SEMILLA)
    pipe = a.armar_pipeline(modelo, random_state=SEMILLA, numeric=[*a.NUMERICAS, variable])
    pipe.steps.insert(1, ("crear", a.CrearVariables()))
    return pipe


base_hgb = cross_val_score(
    a.armar_pipeline(comparados["hist_gb"](), random_state=SEMILLA), X, y, cv=CV, scoring="average_precision"
)
filas = []
for variable in a.VARIABLES_NUEVAS:
    r = cross_val_score(con_una(variable), X, y, cv=CV, scoring="average_precision")
    d = c.comparar_particiones(r, base_hgb, FOLDS)
    filas.append(
        {"variable": variable, "pr_auc": r.mean(), "dif": d["diferencia"], "ic_inf": d["ic_inf"], "ic_sup": d["ic_sup"]}
    )
pd.DataFrame(filas).round(3)

## 5. Variables inspiradas en otros proyectos de abandono

Antes de crear variables se revisaron proyectos y artículos de abandono en banca. Varias ideas necesitan datos que
este extracto no trae. Estas sí se pueden calcular aquí:

| Variable | Fórmula | Por qué probarla |
|---|---|---|
| `EDAD_POR_ANTIGUEDAD` | `EDAD / (1 + ANTIGUEDAD)` | Razón edad-antigüedad usada en un proyecto de abandono bancario |
| `TIPOS_SENAL` | cuántas señales de salida distintas tiene | Aproxima la "amplitud": varias señales distintas frente a una repetida |
| `ALGUNA_SENAL` | 1 si tiene al menos una señal de salida | Separa "nada" de "algo" en un modelo lineal |
| `BALANCE_CREDITO_SALIDA` | `SENALES_SALIDA − #SOLICITUDCREDITOS_ULT3MESES` | Contrasta pedir salir con seguir pidiendo crédito |

Se calculan dentro del pipeline con reglas fijas, sin aprender de los datos, igual que `CrearVariables`.

In [ ]:
INSPIRADAS = ["EDAD_POR_ANTIGUEDAD", "TIPOS_SENAL", "ALGUNA_SENAL", "BALANCE_CREDITO_SALIDA"]


def crear_inspiradas(datos: pd.DataFrame) -> pd.DataFrame:
    """Agrega INSPIRADAS a la salida de CrearVariables (sin estado)."""
    datos = datos.copy()
    datos["EDAD_POR_ANTIGUEDAD"] = datos["EDAD"] / (1 + datos["ANTIGUEDAD"])
    datos["TIPOS_SENAL"] = (datos[a.SALIDA] > 0).sum(axis=1)
    datos["ALGUNA_SENAL"] = (datos["TIPOS_SENAL"] > 0).astype(float)
    datos["BALANCE_CREDITO_SALIDA"] = datos["SENALES_SALIDA"] - datos["#SOLICITUDCREDITOS_ULT3MESES"]
    return datos


def pipeline_inspiradas(modelo, extra):
    """El pipeline del proyecto, con las columnas ``extra`` después de CrearVariables."""
    numericas = [*a.NUMERICAS, *a.VARIABLES_NUEVAS, *extra]
    pipe = a.armar_pipeline(modelo, crear_variables=True, random_state=SEMILLA, numeric=numericas)
    pipe.steps.insert(2, ("inspiradas", FunctionTransformer(crear_inspiradas)))
    return pipe


def medir(pipe, datos=X):
    return cross_val_score(pipe, datos, y, cv=CV, scoring="average_precision")


modelos = {
    "logística": lambda: LogisticRegression(max_iter=2000),
    "hist_gb": lambda: HistGradientBoostingClassifier(random_state=SEMILLA),
}
filas = []
for nombre, modelo in modelos.items():
    referencia = medir(pipeline_inspiradas(modelo(), []))
    filas.append({"modelo": nombre, "variable": "(CrearVariables, referencia)", "pr_auc": referencia.mean()})
    for extra in [[v] for v in INSPIRADAS] + [INSPIRADAS]:
        r = medir(pipeline_inspiradas(modelo(), extra))
        d = c.comparar_particiones(r, referencia, FOLDS)
        filas.append(
            {
                "modelo": nombre,
                "variable": "las 4 juntas" if len(extra) > 1 else extra[0],
                "pr_auc": r.mean(),
                "dif": d["diferencia"],
                "ic_inf": d["ic_inf"],
                "ic_sup": d["ic_sup"],
                "concluyente": d["ic_inf"] > 0 or d["ic_sup"] < 0,
            }
        )
pd.DataFrame(filas).round(3)

## 6. ¿De qué variables depende el modelo?

Esta prueba hace lo contrario: **quita** un grupo de variables y mide cuánto cae la PR-AUC, con la logística sin
balancear del campeón. Si un grupo se vacía por completo, `ColumnDropper` lo elimina dentro del pipeline.

**Por qué importa:** los conteos de comportamiento llegan **sin fecha de medición**. Si el fondo los midió en la
fecha de extracción, para los retirados incluyen hechos posteriores a t0, y eso sería fuga. La fila "sin conteos" es
una aproximación de lo que quedaría si hubiera que descartarlos todos.

In [ ]:
perfil = [v for v in X.columns if v not in a.COMPORTAMIENTO]
escenarios = {
    "todas las variables": [],
    "solo conteos de comportamiento": perfil,
    "sin atributos sin fecha (salario, cargo...)": a.ATRIBUTOS_SIN_FECHA,
    "sin #INTENCIONRETIROS_ULT3MESES": ["#INTENCIONRETIROS_ULT3MESES"],
    "sin señales de salida (a.SALIDA)": a.SALIDA,
    "sin conteos (solo perfil)": a.COMPORTAMIENTO,
}


def sin_columnas(quitar):
    datos = X.copy()
    datos[list(quitar)] = np.nan
    return datos


resultados = {
    nombre: medir(a.armar_pipeline(LogisticRegression(max_iter=2000), random_state=SEMILLA), sin_columnas(quitar))
    for nombre, quitar in escenarios.items()
}
referencia = resultados["todas las variables"]
filas = []
for nombre, r in resultados.items():
    d = c.comparar_particiones(r, referencia, FOLDS)
    filas.append(
        {
            "escenario": nombre,
            "pr_auc": r.mean(),
            "desv": r.std(),
            "dif": d["diferencia"],
            "ic_inf": d["ic_inf"],
            "ic_sup": d["ic_sup"],
        }
    )
dependencia = pd.DataFrame(filas)
display(dependencia.round(3))

fig = px.bar(
    dependencia.sort_values("pr_auc"),
    x="pr_auc",
    y="escenario",
    orientation="h",
    text=dependencia.sort_values("pr_auc")["pr_auc"].map("{:.3f}".format),
    labels={"pr_auc": "PR-AUC (validación cruzada 5 × 3)", "escenario": ""},
    title="PR-AUC según las variables disponibles (datos sintéticos)",
    color_discrete_sequence=[COLOR],
)
fig.add_vline(
    x=BASE_AZAR,
    line={"dash": "dot", "color": "#898781"},
    annotation_text=f"azar = {BASE_AZAR:.3f}",
    annotation_yshift=14,
)
fig.update_traces(textposition="outside", textangle=0)
fig.update_layout(template="plotly_white", xaxis_range=[0, 1.05], margin={"t": 90})
fig

## 7. ¿Cuánto infla las métricas la fuga? Un experimento con fechas

`generador.py` no le pone fecha a las señales de salida: decide primero si el asociado se retira y luego sortea sus
conteos. Por eso no permite medir la fuga. Aquí se usa un **generador nuevo, solo de este notebook**, que le da una
fecha a cada señal. No modifica `generador.py`: solo toma de él los datos del perfil y las fechas de retiro.

**Cómo funciona `GeneradorEventos`:**

- **Eventos de fondo:** cualquier asociado puede tener señales en cualquier fecha, con la tasa de un activo.
- **Eventos antes de irse:** quien se retira recibe señales adicionales hasta completar la tasa de un retirado, con
  fecha `retiro − d`. `d` se sortea con una distribución exponencial cuya media es la **anticipación**.
- **Sin eventos posteriores al retiro**, ni anteriores al ingreso.

No conocemos la anticipación real, así que se prueban varias: de 15 a 180 días.

**Con las mismas señales se miden dos conteos de igual duración (3 meses):**

| Conteo | Ventana | Qué representa |
|---|---|---|
| A t0 | de t0 − 3 meses a t0 | Lo que se sabía el día de la foto: **sin fuga** |
| Al corte | de t0 al corte | Medido en la fecha de extracción: **con fuga** |

**Y se evalúan tres situaciones:**

| Situación | Entrena con | Prueba con | Pregunta que responde |
|---|---|---|---|
| Lo que se reportaría | al corte | al corte | ¿Qué cifra veríamos con fuga? |
| Lo que pasaría en uso | al corte | a t0 | ¿Cómo rinde ese modelo cuando las señales futuras aún no existen? |
| Sin fuga | a t0 | a t0 | ¿Cuánto se puede anticipar de verdad? |

**Para aislar el efecto**, las otras variables de comportamiento (`#SOLICITUDCREDITOS_ULT3MESES`,
`#CANTIDAD_AHORROSCONTRACTUALES` y `#CANTIDAD_SERVICIOS`) se sortean igual para todos: en `generador.py` también
dependen del retiro y traerían la señal sin fecha. El perfil, incluido el tipo de contrato, no se cambia.

> Es una simulación: muestra **cómo funciona** el problema y su orden de magnitud bajo estos supuestos. No mide la fuga
> del fondo.

In [ ]:
from sklearn.metrics import average_precision_score

from proyecto_ml.evaluate import recall_at_k

# Señales por cada 3 meses: (activo, retirado). Son las mismas tasas de generador.py.
TASAS_SENALES = {
    "#SOLICITUD_DISMINUCIONCUOTAS": (0.3, 1.5),
    "#SOLICITUD_CANCELACIONAHORROS": (0.1, 1.2),
    "#SOLICITUD_RETIROPARCIALAHORROS": (0.6, 2.0),
    "CANCELACIONCREDITOS_ULT3MESES": (0.1, 0.8),
    "#INTENCIONRETIROS_ULT3MESES": (0.2, 1.5),
}
T0 = a.fecha_observacion()
INICIO = T0 - pd.DateOffset(months=a.HORIZONTE_MESES)
HISTORIA = T0 - pd.DateOffset(months=6)  # desde cuándo existen eventos (para los conteos acumulados)


class GeneradorEventos:
    """Le da una fecha a cada señal de salida de los asociados de una foto (no reemplaza a GeneradorAsociados)."""

    def __init__(self, anticipacion_dias: float, seed: int = 0):
        self.anticipacion_dias = anticipacion_dias
        self.seed = seed

    def generar(self, foto: pd.DataFrame) -> pd.DataFrame:
        """Una fila por evento: asociado, senal, fecha y origen ("fondo" o "antes de irse")."""
        rng = np.random.default_rng(self.seed)
        retiro = pd.to_datetime(foto["FECHARETIRO"])
        ingreso = pd.to_datetime(foto["FECHAINGRESOLABORA"])
        retirados = foto.index[foto[a.TARGET] == 1]
        dias = (a.FECHA_CORTE - HISTORIA).days
        partes = []
        for senal, (activo, retirado) in TASAS_SENALES.items():
            # De fondo: 9 meses (de HISTORIA al corte, 3 ventanas de 3 meses), con fecha uniforme.
            ids = np.repeat(foto.index, rng.poisson(3 * activo, len(foto)))
            fechas = HISTORIA + pd.to_timedelta(rng.uniform(0, dias, len(ids)), unit="D")
            partes.append(pd.DataFrame({"asociado": ids, "senal": senal, "fecha": fechas, "origen": "fondo"}))
            # Antes de irse: lo que falta para llegar a la tasa de un retirado, contado hacia atrás desde el retiro.
            ids = np.repeat(retirados, rng.poisson(retirado - activo, len(retirados)))
            antes = pd.to_timedelta(rng.exponential(self.anticipacion_dias, len(ids)), unit="D")
            fechas = retiro.loc[ids].to_numpy() - antes
            partes.append(pd.DataFrame({"asociado": ids, "senal": senal, "fecha": fechas, "origen": "antes de irse"}))
        eventos = pd.concat(partes, ignore_index=True)
        tras_retiro = eventos["fecha"] > eventos["asociado"].map(retiro)  # NaT da False: los activos siguen
        antes_de_ingreso = eventos["fecha"] < eventos["asociado"].map(ingreso)
        return eventos[~tras_retiro & ~antes_de_ingreso].reset_index(drop=True)


def contar(eventos: pd.DataFrame, indice: pd.Index, desde: pd.Timestamp, hasta: pd.Timestamp) -> pd.DataFrame:
    """Conteo de cada señal por asociado en la ventana (desde, hasta]. Sin eventos, 0."""
    ventana = eventos[(eventos["fecha"] > desde) & (eventos["fecha"] <= hasta)]
    tabla = pd.crosstab(ventana["asociado"], ventana["senal"])
    return tabla.reindex(index=indice, columns=list(TASAS_SENALES), fill_value=0).astype(float)


def variables_con_fechas(
    eventos: pd.DataFrame, seed: int, acumulado: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Las variables de la foto con los conteos medidos a t0 y al corte. Lo demás es idéntico en ambas.

    Por defecto, ventanas de 3 meses. Con ``acumulado``, todo lo ocurrido desde HISTORIA hasta cada fecha.
    """
    rng = np.random.default_rng(seed + 1000)
    base = X.copy()
    base["#SOLICITUDCREDITOS_ULT3MESES"] = rng.poisson(0.9, len(base)).astype(float)
    base["#CANTIDAD_AHORROSCONTRACTUALES"] = rng.poisson(1.0, len(base)).astype(float)
    base["#CANTIDAD_SERVICIOS"] = rng.integers(1, 7, len(base)).astype(float)
    a_t0, al_corte = base.copy(), base.copy()
    a_t0[list(TASAS_SENALES)] = contar(eventos, base.index, HISTORIA if acumulado else INICIO, T0)
    al_corte[list(TASAS_SENALES)] = contar(eventos, base.index, HISTORIA if acumulado else T0, a.FECHA_CORTE)
    return a_t0, al_corte


ejemplo = GeneradorEventos(anticipacion_dias=30, seed=0).generar(vista)
print(f"{len(ejemplo)} eventos con fecha, anticipación de 30 días. Un asociado que se retira:")
primero = y[y == 1].index[0]
print(f"Retiro: {pd.Timestamp(vista.loc[primero, 'FECHARETIRO']).date()} · t0: {T0.date()}")
ejemplo[ejemplo["asociado"] == primero].sort_values("fecha").assign(fecha=lambda d: d["fecha"].dt.date)

### Comprobación: ¿el generador hace lo que dice?

Antes de usarlo, conviene revisar el promedio de señales de salida por clase y por ventana. Con una anticipación corta,
las señales de quien se va deberían caer casi todas después de t0. Con una larga, deberían repartirse.

In [ ]:
ANTICIPACIONES = [15, 30, 60, 90, 180]
filas = []
for dias in ANTICIPACIONES:
    eventos = GeneradorEventos(dias, seed=0).generar(vista)
    a_t0, al_corte = variables_con_fechas(eventos, seed=0)
    extra = eventos[eventos["origen"] == "antes de irse"]
    filas.append(
        {
            "anticipacion_dias": dias,
            "retirados_a_t0": a_t0.loc[y == 1, list(TASAS_SENALES)].sum(axis=1).mean(),
            "retirados_al_corte": al_corte.loc[y == 1, list(TASAS_SENALES)].sum(axis=1).mean(),
            "activos_a_t0": a_t0.loc[y == 0, list(TASAS_SENALES)].sum(axis=1).mean(),
            "activos_al_corte": al_corte.loc[y == 0, list(TASAS_SENALES)].sum(axis=1).mean(),
            "extra_despues_de_t0": (extra["fecha"] > T0).mean(),
        }
    )
pd.DataFrame(filas).round(2)

### La medición

Se repite con 3 semillas de eventos y validación cruzada 5 × 3, con la logística sin balancear del campeón. Cada modelo
se ajusta solo con las filas de entrenamiento de cada partición.

Además de la PR-AUC se mide el **recall en el 10 %**: de quienes se retiran, cuántos aparecen en el 10 % de asociados con
mayor puntaje. Es más fácil de explicar: "si el equipo llama a 460 personas, ¿a cuántos de los que se van encuentra?".

La **referencia** usa los conteos sin fecha de `generador.py`, en las mismas condiciones. Es decir, la suposición actual
de que la señal ya es visible en t0.

In [ ]:
SEMILLAS_EVENTOS = (0, 1, 2)


def evaluar(entrena: pd.DataFrame, prueba: pd.DataFrame, cv) -> dict:
    """Entrena con una versión de las variables y prueba con otra, en las mismas particiones."""
    pr_auc, recall = [], []
    for i_train, i_test in cv.split(entrena, y):
        pipe = a.armar_pipeline(LogisticRegression(max_iter=2000), random_state=SEMILLA)
        pipe.fit(entrena.iloc[i_train], y.iloc[i_train])
        puntaje = pipe.predict_proba(prueba.iloc[i_test])[:, 1]
        pr_auc.append(average_precision_score(y.iloc[i_test], puntaje))
        recall.append(recall_at_k(y.iloc[i_test], puntaje))
    return {"pr_auc": np.mean(pr_auc), "recall_10": np.mean(recall)}


filas = []
for dias in ANTICIPACIONES:
    for semilla in SEMILLAS_EVENTOS:
        eventos = GeneradorEventos(dias, seed=semilla).generar(vista)
        a_t0, al_corte = variables_con_fechas(eventos, semilla)
        situaciones = {
            "Lo que se reportaría": (al_corte, al_corte),
            "Lo que pasaría en uso": (al_corte, a_t0),
            "Sin fuga": (a_t0, a_t0),
        }
        for situacion, (entrena, prueba) in situaciones.items():
            filas.append(
                {"anticipacion": dias, "semilla": semilla, "situacion": situacion} | evaluar(entrena, prueba, CV)
            )
fuga = pd.DataFrame(filas)

referencia = []
for semilla in SEMILLAS_EVENTOS:
    sin_fecha = variables_con_fechas(GeneradorEventos(30, seed=semilla).generar(vista), semilla)[0]
    sin_fecha[list(TASAS_SENALES)] = X[list(TASAS_SENALES)]  # los conteos sin fecha de generador.py
    referencia.append(evaluar(sin_fecha, sin_fecha, CV))
REFERENCIA = pd.DataFrame(referencia).mean()
print(f"Referencia, conteos sin fecha: PR-AUC {REFERENCIA['pr_auc']:.3f}, recall 10 % {REFERENCIA['recall_10']:.3f}")

resumen_fuga = (
    fuga.groupby(["anticipacion", "situacion"], sort=False)
    .agg(pr_auc=("pr_auc", "mean"), pr_min=("pr_auc", "min"), pr_max=("pr_auc", "max"), recall_10=("recall_10", "mean"))
    .reset_index()
)
display(resumen_fuga.round(3))
resumen_fuga.pivot_table(index="anticipacion", columns="situacion", values="pr_auc").round(3)

### El resultado

La distancia entre **lo que se reportaría** y **lo que pasaría en uso** es el tamaño del engaño: el equipo vería la
primera línea, y el fondo recibiría la segunda.

Con 60 días o más, las líneas se cruzan: en ventanas de 3 meses, medir al corte deja fuera las señales anteriores a t0,
así que la cifra con fuga baja. La variante con conteos acumulados, más abajo, no tiene ese efecto.

In [ ]:
ORDEN = ["Lo que se reportaría", "Sin fuga", "Lo que pasaría en uso"]
COLORES_FUGA = dict(zip(ORDEN, ["#2a78d6", "#eb6834", "#1baf7a"], strict=True))  # paleta de referencia, en orden fijo
grafico = resumen_fuga.assign(anticipacion=resumen_fuga["anticipacion"].astype(str))
fig = px.line(
    grafico,
    x="anticipacion",
    y="pr_auc",
    color="situacion",
    markers=True,
    category_orders={"situacion": ORDEN, "anticipacion": [str(d) for d in ANTICIPACIONES]},
    color_discrete_map=COLORES_FUGA,
    hover_data={"recall_10": ":.2f", "pr_min": ":.3f", "pr_max": ":.3f"},
    labels={
        "anticipacion": "Anticipación media de las señales antes del retiro (días)",
        "pr_auc": "PR-AUC",
        "situacion": "",
        "recall_10": "Recall en el 10 %",
        "pr_min": "Mínimo entre semillas",
        "pr_max": "Máximo entre semillas",
    },
    title="La fuga infla la cifra reportada cuando las señales llegan cerca del retiro (simulación)",
)
fig.update_traces(line={"width": 2}, marker={"size": 9, "line": {"width": 2, "color": "#fcfcfb"}})
fig.add_hline(
    y=BASE_AZAR,
    line={"dash": "dot", "width": 1, "color": "#898781"},
    annotation_text=f"Azar: {BASE_AZAR:.3f}",
    annotation_position="top left",
)
fig.add_hline(
    y=REFERENCIA["pr_auc"],
    line={"dash": "dot", "width": 1, "color": "#898781"},
    annotation_text=f"Conteos sin fecha: {REFERENCIA['pr_auc']:.3f}",
    annotation_position="top right",
)
fig.update_yaxes(range=[0, 1.05], gridcolor="#eeeeea")
fig.update_xaxes(type="category", showgrid=False)
fig.update_layout(template="plotly_white", legend={"orientation": "h", "y": -0.2}, height=480)
fig

### ¿Y si los conteos son acumulados?

La tabla anterior compara ventanas de 3 meses. Con ellas, medir al corte **pierde** lo ocurrido antes de t0, y por eso la
cifra "con fuga" puede salir menor que la cifra sin fuga cuando las señales llegan con mucha anticipación.

Pero no todas las columnas dicen `ULT3MESES`: `#SOLICITUD_CANCELACIONAHORROS`, por ejemplo, podría contar todo el historial.
Aquí se repite la medición con conteos acumulados, de modo que medir al corte solo **agrega** información: 6 meses de
historia hasta t0 frente a 9 meses hasta el corte.

In [ ]:
filas = []
for dias in ANTICIPACIONES:
    for semilla in SEMILLAS_EVENTOS:
        eventos = GeneradorEventos(dias, seed=semilla).generar(vista)
        a_t0, al_corte = variables_con_fechas(eventos, semilla, acumulado=True)
        situaciones = {
            "Lo que se reportaría": (al_corte, al_corte),
            "Lo que pasaría en uso": (al_corte, a_t0),
            "Sin fuga": (a_t0, a_t0),
        }
        for situacion, (entrena, prueba) in situaciones.items():
            filas.append(
                {"anticipacion": dias, "semilla": semilla, "situacion": situacion} | evaluar(entrena, prueba, CV)
            )
fuga_acumulada = pd.DataFrame(filas)
tablas_acumuladas = {
    medida: fuga_acumulada.pivot_table(index="anticipacion", columns="situacion", values=medida).round(3)
    for medida in ("pr_auc", "recall_10")
}
display(tablas_acumuladas["pr_auc"])
tablas_acumuladas["recall_10"]

## 8. Hallazgos (para la presentación del 11-nov)

*Todas las cifras son de datos sintéticos.*

1. **Sumar las señales ayuda a los árboles, no a la logística.** Con `hist_gb`, `CrearVariables` sube la PR-AUC en los
   4 datasets (+0,024 a +0,094), aunque solo es concluyente en 2. Con la logística queda entre −0,004 y 0,000.
2. **La explicación probable:** una logística ya suma sus variables con pesos, mientras que un árbol corta por una
   variable a la vez y necesita muchos cortes para aproximar una suma. La mejora viene casi toda de `SENALES_SALIDA`.
3. **`SALARIO_LOG` no cambia nada en los árboles** (dif. 0,000): cortar por salario o por su logaritmo es lo mismo.
4. **La logística sigue ganando:** en los 4 datasets supera a `hist_gb`, incluso cuando este usa las variables nuevas.
   Por eso estas variables no cambian al campeón.
5. **Las 4 variables inspiradas en otros proyectos no aportan** de forma concluyente con ninguno de los dos modelos.
6. **El mayor riesgo está en los datos, no en las variables.** La PR-AUC pasa de 0,915 a 0,048 sin los conteos sin
   fecha (azar: 0,023). Si el fondo los midió en la fecha de extracción, hay que replantear las variables.

**Lo que agrega el experimento con fechas (sección 7):**

7. **La cifra de 0,88 depende de una suposición.** Con los conteos sin fecha, y en las mismas condiciones del experimento,
   la PR-AUC es 0,883. Cuando las señales tienen fecha, la PR-AUC sin fuga queda entre 0,10 y 0,38, según su anticipación y
   la forma de contar. Sigue superando al azar (0,023), pero está lejos de 0,88.
8. **La fuga infla más cuando las señales llegan cerca del retiro.** Con 15 días de anticipación y ventanas de 3 meses, se
   reportaría 0,632, y en uso el modelo daría 0,154. Con conteos acumulados y 30 días, serían 0,542 frente a 0,207.
9. **La forma de contar cambia el resultado.** Con conteos acumulados, la cifra con fuga siempre supera a la cifra en uso.
   Con ventanas de 3 meses y 60 días o más, sale incluso por debajo, porque medir al corte deja fuera lo anterior a t0.
   Medir mal no siempre sube la cifra: también puede esconder señal.
10. **El resultado más útil es de negocio.** Sin fuga, el 10 % con mayor puntaje encuentra entre el 29 % y el 65 % de los que
    se retiran, frente al 95 % de la referencia. Cuánto se puede anticipar depende de cuánto antes aparezcan las señales.

**Límites del experimento:** las tasas, la distribución exponencial y las ventanas son supuestos, y se usaron 3 semillas de
eventos. No simula a quienes se retiran después del corte. Muestra el mecanismo y su orden de magnitud, no la fuga del fondo.

**Límite de estas conclusiones:** el generador pone la señal en los conteos y en el tipo de contrato. Estos datos
pueden mostrar cómo se comporta cada modelo, pero no qué variables servirán con el fondo real.

## 9. Qué pedirle al fondo

En los proyectos de abandono revisados destacan el **comportamiento reciente y sus cambios**, algo que este extracto
no puede medir porque sus conteos no tienen fecha.

**Evidencia encontrada:**

- [Brito et al. (2024)](https://link.springer.com/article/10.1186/s40854-023-00558-3), banca minorista, más de 3
  millones de clientes: presentan variables de recencia, frecuencia y valor monetario (RFM) como variables con
  capacidad superior para predecir el abandono, y usan PR-AUC como métrica principal. No pude leer el texto completo
  ni sus cifras por variable.
- [Análisis de BankChurners](https://leekarensl.github.io/2023/03/24/customer-churn-prediction.html): las variables
  más importantes son el número y el monto de transacciones, y el cambio entre trimestres (`Total_Ct_Chng_Q4_Q1`). No
  son variables creadas por el autor y no compara antes y después.
- [Sound Credit Union](https://bpb-us-e1.wpmucdn.com/sites.uw.edu/dist/3/652/files/2020/05/Showcase-2022_Project_Poster_Team-7_SoundCreditUnion.pdf):
  afirma que el comportamiento transaccional refinó el modelo y usa modelos separados para quienes tienen créditos y
  quienes tienen ahorros. Es un póster, sin cifras por variable.
- [Razón edad-antigüedad](https://github.com/sayedgamal99/predicting-bank-customer-churn-using-machine-learning):
  un proyecto de GitHub la crea, pero no mide cuánto aporta. Aquí tampoco aportó (sección 5).

**Ninguna fuente revisada mide con y sin cada variable**, así que esto orienta qué probar, pero no garantiza una mejora.

**Variables para probar si el fondo entrega movimientos con fecha.** Todas se calcularían solo con datos hasta t0:

| Tipo | Variable | Dato necesario |
|---|---|---|
| Recencia | Días desde el último aporte, crédito o solicitud | Fecha de cada movimiento |
| Frecuencia | Movimientos en los 3 meses antes de t0 | Fecha de cada movimiento |
| Valor | Monto de aportes y saldo de ahorro a t0 | Montos y saldos con fecha |
| Cambio | Aportes de los últimos 3 meses / los 3 meses anteriores | Dos ventanas antes de t0 |
| Cambio | Variación del saldo de ahorro y de crédito | Saldos mensuales |

Las fechas también resolverían la duda de la fuga: cada conteo se mediría a t0.